# 論文情報

#### 題名
[リレーションシップバンキング機能の向上を目的とした中小企業の資金ニーズ判別法とその活用の提案](https://infoshako.sk.tsukuba.ac.jp/~databank/pdf/1385.pdf)

#### 概要
リレーションシップバンキングは、金融機関が顧客と親密な関係を築き、長期的な取引を行うことで収益を得るビジネスモデルである。中小・地域金融機関はこの機能を有し、中小企業の資金調達を支える役割が期待されるが、近年は過度な取引先負担やIT技術の活用不足が課題となっている。本研究では、信用金庫内で秘匿化データを用いた解析を行い、中小企業の資金ニーズを80%以上の精度で判別するモデルを開発した。このモデルは、金融機関のリレーションシップバンキング業務の効率化に貢献し得る。

#### 背景
金融機関のビジネスモデルは「トランザクションバンキング」と「リレーションシップバンキング」に分類される。大銀行はトランザクションバンキングを志向する一方、中小・地域金融機関はリレーションシップバンキングを重視している。しかし、近年は取引先への過度なコミットメントコストが金融機関の収益性や健全性の低下を招いており、リレーションシップバンキングの効率化が求められている。

#### 方法
本研究は、水戸信用金庫の秘匿化データを用いて、中小企業の資金ニーズを予測する機械学習モデルを構築した。2016年1月から2021年3月までの27,068事業者の財務データを分析し、財務諸表の情報から長期借入金の増減を判別するモデルを開発。標準化、主成分分析（SVD）、ロジスティック回帰を組み合わせた手法を適用し、精度の向上を図った。

#### 結果
提案したモデルは、資金ニーズを80%以上の精度で判別できることが確認された。売上の増減による企業分類を行うことで、さらに精度が向上し、AUC（ROC曲線下面積）は0.75以上を記録。特に、売上増加企業と売上減少企業に分けた場合の判別性能は高かった。

#### モデル概要
・モデル：リッジ・ロジスティック回帰(scikit-learn.linear_model.LogisticRegression)  
・被説明変数：（t+1年度の長期借入金）/（t年度の長期借入金） >=1 →1  
・説明変数：462個  
| 説明変数の種類 | 変数の個数 | 例 |
|--------------|----------|----------------------|
| `𝑡-1` 年度の財務諸表の値 | 194個 | 貸借対照表項目、損益計算書項目、キャッシュフロー計算書項目 |
| `𝑡` 年度の財務諸表の値 | 194個 | 同上 |
| `𝑡` 年度の値 - `𝑡-1` 年度の値（変化量） | 194個 | 収益の増減、負債の増減など |
| `𝑡-1` 年度の財務比率 | 193個 | **貸借対照表項目:** `𝑡-1` 年度の財務諸表の値 / `𝑡-1` 年度の総資産額 **それ以外:** `𝑡-1` 年度の財務諸表の値 / `𝑡-1` 年度の売上高 |
| `𝑡` 年度の財務比率 | 193個 | 同上 |
| `𝑡` 年度の比 - `𝑡-1` 年度の比（比率変化） | 193個 | 各比率の増減 |
| `𝑡-1` 年度から `𝑡` 年度にかけての売上増減ダミー変数 | 1個 | **1:** 売上増加 **0:** 売上減少 |
| 年度ダミー変数 | 4個 | 各年度ごとの影響を考慮 |


#### データ前処理
・標準化  
・特異値分解(450次元)  
・重要度の高い変数のみ抽出  

#### 評価
・accuracy：正解率  
・ROC  
・閾値設定

# 実装コード

#### データ作成

In [ ]:
import os
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from program1 import process_financial_data
from program2 import process_intermediate_data

In [ ]:
# 入力・出力ファイルのパスを設定
input_file = "financial_data.csv"
intermediate_file = "intermediate_data.csv"
output_file = "final_data.csv"

# ファイルの存在確認
if not os.path.exists(input_file):
    raise FileNotFoundError(f"入力ファイル {input_file} が見つかりません。")

try:
    # Program1の実行
    print("Step 1: 財務データの前処理を開始...")
    process_financial_data(input_file, intermediate_file)
    print(f"中間ファイル {intermediate_file} を作成しました。")

    # Program2の実行
    print("\nStep 2: 特徴量エンジニアリングを開始...")
    process_intermediate_data(intermediate_file, output_file)
    print(f"最終ファイル {output_file} を作成しました。")

    # 中間ファイルの削除（オプション）
    os.remove(intermediate_file)
    print(f"中間ファイル {intermediate_file} を削除しました。")

    print("\n処理が正常に完了しました。")

except Exception as e:
    print(f"\nエラーが発生しました: {str(e)}")
    raise

#### データ加工

In [ ]:
# データの読み込み
data = pd.read_csv("final_data.csv")

# 総資産額、売上高のいずれかが欠損もしくは0のレコードを削除
data = data[(data['総資産額'].notna()) & (data['総資産額'] != 0) & 
            (data['売上高'].notna()) & (data['売上高'] != 0)]

# 数値カラムの中で分散が0のカラムを削除
data = data.loc[:, data.var() != 0]

# 各カラムの欠損値の割合が10%以上のカラムを削除
threshold = 0.1
data = data.loc[:, data.isnull().mean() < threshold]

# 各レコードの欠損値の割合が10%以上のレコードを削除
data = data.loc[data.isnull().mean(axis=1) < threshold]

# 欠損値を0で補完
data = data.fillna(0)

# 加工後のデータを保存
data.to_csv("processed_data.csv", index=False)
print("データ加工が完了しました。")

#### 被説明変数結合

#### モデル作成

In [ ]:
# データの読み込み
data = pd.read_csv("final_data.csv")

# 説明変数と被説明変数の分割
X = data.drop("target", axis=1)
y = data["target"]

# 数値データの標準化
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# カテゴリデータのダミー化
categorical_features = X.select_dtypes(include=['object']).columns
categorical_transformer = Pipeline(steps=[
    ('onehot', pd.get_dummies(X[categorical_features]))
])

In [ ]:
# 前処理の適用
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# SVDで次元を410までに圧縮
svd = TruncatedSVD(n_components=410)

# 目的変数と相関の強いものを100個選択
selector = SelectKBest(score_func=f_classif, k=100)

# リッジ・ロジスティック回帰
model = LogisticRegression(penalty='l2', solver='liblinear')

# パイプラインの作成
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('svd', svd),
                           ('selector', selector),
                           ('classifier', model)])

# データの分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# モデルの学習
pipeline.fit(X_train, y_train)

# 予測
y_pred = pipeline.predict(X_test)

#### 評価

In [ ]:
# 評価
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"ROC AUC: {roc_auc}")